In [72]:
# from pydantic import BaseModel
from typing import List, Optional
import random
import torch
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, TaskType, PeftModel
from datasets import Dataset

In [49]:
MODEL_DIR = "./latest_trained_model"
TOKENIZER = None
MODEL = None

In [50]:
# !huggingface-cli download deepseek-ai/DeepSeek-R1-Distill-Llama-8B \
#   --local-dir ./deepseek-model \
#   --local-dir-use-symlinks False
# model_name = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
model_name = './deepseek-model'
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
    device_map="auto",
    low_cpu_mem_usage=True
)

In [51]:
model = AutoModelForCausalLM.from_pretrained(
    "./deepseek-model",
    device_map="auto",
    low_cpu_mem_usage=True
)
prompt = "인공지능이란?"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


인공지능이란? | AI is what?
AI stands for Artificial Intelligence, which is the simulation of human intelligence in machines that are programmed to think and learn like humans. It encompasses various technologies like machine learning, natural language processing, and robotics.

AI can perform tasks that


In [52]:
bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map={"": 0},
    low_cpu_mem_usage=False
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05
)
model = get_peft_model(model, lora_config)


oading checkpoint shards: 100%|██████████| 2/2 [00:11<00:00,  5.57s/it]

In [53]:
model.print_trainable_parameters()

trainable params: 3,407,872 || all params: 8,033,669,120 || trainable%: 0.0424


In [75]:
question_templates = [
    "한국의 수도는 어디인가요?",
    "파이썬은 어떤 언어인가요?",
    "봄에 여행하기 좋은 곳은?",
    "로라는 무엇인가요?",
    "AI의 장점은 무엇인가요?",
    "코딩을 배우려면 어떻게 시작하나요?",
    "고양이가 사람을 좋아하는 이유는?",
    "서울의 명소를 추천해주세요.",
    "시간 여행이 가능할까요?",
    "지구는 어떤 행성인가요?"
]

answer_templates = [
    "서울입니다.",
    "파이썬은 배우기 쉬운 프로그래밍 언어입니다.",
    "제주도나 경주가 봄 여행지로 좋습니다.",
    "LoRA는 대형 언어 모델을 효율적으로 미세조정하는 기술입니다.",
    "AI는 빠르고 정확한 분석이 가능합니다.",
    "온라인 강의와 예제를 따라 실습하는 것이 좋습니다.",
    "고양이는 사람에게 애정을 느낄 수 있습니다.",
    "경복궁, 남산타워, 한강공원 등이 있습니다.",
    "현재 기술로는 어렵지만 이론적으로 연구는 진행 중입니다.",
    "지구는 생명체가 사는 유일한 행성입니다."
]

data = []
for i in range(200):
    q = random.choice(question_templates)
    a = random.choice(answer_templates)
    data.append({"input": q, "output": a})

dataset = Dataset.from_list(data)

print(dataset[99])

{'input': 'AI의 장점은 무엇인가요?', 'output': '서울입니다.'}


In [76]:
def preprocess(example):
    prompt = example["input"]
    response = example["output"]
    full_input = f"{prompt}\n### 답변:\n"
    full_text = full_input + response

    tokenized = tokenizer(
        full_text,
        truncation=True,
        padding="max_length",
        max_length=256,
        return_tensors="pt"
    )
    
    input_ids = tokenized["input_ids"][0]
    attention_mask = tokenized["attention_mask"][0]
    labels = input_ids.clone()
    prompt_len = len(tokenizer(full_input, return_tensors="pt")["input_ids"][0])
    labels[:prompt_len] = -100 
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

tokenized_dataset = dataset.map(preprocess)


ap: 100%|██████████| 200/200 [00:00<00:00, 1991.73 examples/s]

In [77]:
epochs = 3

training_args = TrainingArguments(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=epochs,
    # learning_rate=5e-5,
    learning_rate=1e-5,
    logging_steps=10,
    output_dir=MODEL_DIR,
    save_strategy="epoch",
    save_total_limit=2,
    bf16=False,
    fp16=True
)

In [78]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [79]:
trainer.train()

Step,Training Loss
10,0.000000


TrainOutput(global_step=18, training_loss=0.0, metrics={'train_runtime': 89.5872, 'train_samples_per_second': 6.697, 'train_steps_per_second': 0.201, 'total_flos': 6089317759844352.0, 'train_loss': 0.0, 'epoch': 2.64})

In [81]:
model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

('./latest_trained_model/tokenizer_config.json',
 './latest_trained_model/special_tokens_map.json',
 './latest_trained_model/tokenizer.json')

In [82]:
TOKENIZER = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    trust_remote_code=True,
    quantization_config=BitsAndBytesConfig(
        trust_remote_code=True,
        device_map="auto",
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
)
MODEL = PeftModel.from_pretrained(base_model, MODEL_DIR).eval()


oading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.94s/it]

In [83]:
input_text = f"질문은뭐여\n### 답변:\n"
inputs = TOKENIZER(input_text, return_tensors="pt")


device = MODEL.base_model.hf_device_map.get("model.embed_tokens", "cuda:0")
inputs = {k: v.to(device) for k, v in inputs.items()}

In [85]:
with torch.no_grad():
    output = MODEL(**inputs)
    logits = output.logits
    print(torch.isnan(logits).any(), torch.isinf(logits).any(), (logits < 0).any())
with torch.no_grad():
    outputs = MODEL.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.7,
        top_p=0.9,
        do_sample=False
    )
result = TOKENIZER.decode(outputs[0], skip_special_tokens=True)
answer = result.split("### 답변:")[-1].strip()

/home/dicia/dicia/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/dicia/dicia/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


tensor(True, device='cuda:0') tensor(False, device='cuda:0') tensor(False, device='cuda:0')


In [ ]:
result

'!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!'